In [ ]:
# ==============================================================================
# Lab 1: AI for Social Good - Plant Disease Detection
# Ultra-Light Version to prevent RAM crash in Google Colab
# ==============================================================================

# Part 1: Setup & Data Exploration
# ------------------------------------------------------------------------------
print("--- Part 1: Setup & Data Exploration ---")

# Step 1.1: Import necessary libraries
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np

print(f"TensorFlow Version: {tf.__version__}")
print("-" * 30)


# Step 1.2: Load the PlantVillage dataset
# MODIFIED: Using a smaller slice of data (30%) to save RAM
(ds_train, ds_validation, ds_test), ds_info = tfds.load(
    'plant_village',
    split=['train[:30%]', 'train[30%:40%]', 'train[40%:50%]'], # Use 30% train, 10% val, 10% test
    with_info=True,
    as_supervised=True,
)

# Step 1.3: Explore the data
num_classes = ds_info.features['label'].num_classes
class_names = ds_info.features['label'].names

print(f"Number of classes: {num_classes}")
print(f"Number of training examples: {tf.data.experimental.cardinality(ds_train).numpy()}")
print(f"Number of validation examples: {tf.data.experimental.cardinality(ds_validation).numpy()}")
print(f"Number of test examples: {tf.data.experimental.cardinality(ds_test).numpy()}")
print("-" * 30)

# Display some sample images from the training set
print("Displaying sample images...")
plt.figure(figsize=(10, 10))
for i, (image, label) in enumerate(ds_train.take(9)):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(image)
    plt.title(class_names[label])
    plt.axis("off")
plt.show()


In [ ]:


# Part 2: Data Preprocessing
# ------------------------------------------------------------------------------
print("\n--- Part 2: Data Preprocessing ---")

# Step 2.1: Define image size and batch size
IMG_SIZE = 128
BATCH_SIZE = 32

# Step 2.2: Create a preprocessing function
def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = image / 255.0  # Normalize to [0,1]
    return image, label

# Apply the preprocessing function
ds_train = ds_train.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
ds_validation = ds_validation.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
ds_test = ds_test.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
print("Resizing and normalization complete.")


# Configure datasets for performance
# MODIFIED: Removed .cache() to save RAM. This might make training a bit slower
# but prevents crashing.
ds_train = ds_train.shuffle(1000).batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)
ds_validation = ds_validation.batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)
ds_test = ds_test.batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)
print("Datasets configured for performance (RAM optimized).")


In [ ]:
# Part 3: Model Building & Training
# ------------------------------------------------------------------------------
print("\n--- Part 3: Model Building & Training ---")

# Step 3.1: Build the ULTRA-LIGHT CNN model
# MODIFIED: Even simpler model to use less memory and compute power
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    # Convolutional Block 1
    tf.keras.layers.Conv2D(8, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),

    # Convolutional Block 2
    tf.keras.layers.Conv2D(16, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),

    # Flatten and Dense layers
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),

    # Output layer
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

# Step 3.2: Compile the model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# Step 3.3: Train the model
print("\nStarting model training...")
EPOCHS = 5
history = model.fit(
    ds_train,
    validation_data=ds_validation,
    epochs=EPOCHS
)
print("Model training complete.")


In [ ]:
# Part 4: Model Evaluation
# ------------------------------------------------------------------------------
print("\n--- Part 4: Model Evaluation ---")

# Step 4.1: Plot training and validation graphs
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(EPOCHS)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

# Step 4.2: Evaluate the model on the test set
print("\nEvaluating model on the test set...")
test_loss, test_accuracy = model.evaluate(ds_test)
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print("-" * 30)
print("Lab complete!")
